# MNIST RBF-SVM Experiment
使用 RBF 核的支持向量机在 MNIST 上训练，并结合 `graph_print_analysis` 中的工具对精度曲线进行平滑分析。

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import graph_print_analysis as gp_tool
from train_mnist_svm import load_mnist_data, train_rbf_svm, evaluate_accuracy, SVMConfig

# 保证项目根目录在路径里，方便在笔记本内导入
repo_root = Path('..').resolve()
import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


In [ ]:
# 载入数据（为了速度只取 4000 个样本），并划分出测试集
X_train, X_test, y_train, y_test = load_mnist_data(max_samples=4000, test_size=0.2)
X_train.shape, X_test.shape

In [ ]:
# 训练 RBF 核 SVM
config = SVMConfig(c=5.0, gamma='scale')
svm_model = train_rbf_svm(X_train, y_train, config=config)
test_accuracy = evaluate_accuracy(svm_model, X_test, y_test)
print(f'Test accuracy: {test_accuracy:.4f}')

In [ ]:
# 自定义一个简单的准确率函数，并验证与 evaluate_accuracy 一致
def accuracy(y_true, y_pred):
    return float(np.mean(y_true == y_pred))

pred_labels = svm_model.predict(X_test)
print('Accuracy (evaluate_accuracy):', evaluate_accuracy(svm_model, X_test, y_test))
print('Accuracy (custom fn):      ', accuracy(y_test, pred_labels))

In [ ]:
# 使用 graph_print_analysis 平滑累积准确率曲线，观察预测稳定性
correct_flags = (pred_labels == y_test).astype(float)
steps = np.arange(1, len(correct_flags) + 1)
cumulative_acc = np.cumsum(correct_flags) / steps
x_smooth, acc_smooth = gp_tool.moving_average_xy(cumulative_acc, window=50)

plt.figure(figsize=(8, 4))
plt.plot(steps, cumulative_acc, alpha=0.4, label='Cumulative accuracy')
if len(acc_smooth) > 0:
    plt.plot(x_smooth + 1, acc_smooth, label='Smoothed (window=50)', linewidth=2)
plt.xlabel('Test sample index')
plt.ylabel('Accuracy')
plt.title('MNIST RBF-SVM accuracy curve')
plt.legend()
plt.tight_layout()
plt.show()
